# 1. CLONE REPO

In [ ]:
%cd /kaggle/working
!git clone https://github.com/PhuongThao-2005/LViT.git

%cd /kaggle/working/LViT
!git checkout BTRXD-LViT-T-SlidingTrain

# 2. COPY DATASET FULLSIZE

In [ ]:
!cp -r /kaggle/input/datasets/phuongthao205/btrxd-fullsize/BTRXD_fullsize /kaggle/working/LViT/datasets/

# 3. INSTALL PACKAGES

In [ ]:
!pip install -q openpyxl tensorboardX thop transformers ml-collections scipy scikit-learn

# 4. CONFIG

In [ ]:
%%writefile /kaggle/working/LViT/Config.py
# -*- coding: utf-8 -*-
import os, torch, time, ml_collections

save_model = True
tensorboard = True
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
use_cuda = torch.cuda.is_available()
seed = 666
os.environ['PYTHONHASHSEED'] = str(seed)

cosineLR = True
n_channels = 3
n_labels = 1
epochs = 200
img_size = 224
print_frequency = 50
save_frequency = 10
vis_frequency = 50
early_stopping_patience = 50   # 50 × 10 = 500 epoch mới stop (val mỗi 10 epoch)
pretrain = False

task_name = 'BTRXD_fullsize'

learning_rate = 3e-4
batch_size = 16              # AMP + patch nhỏ → tăng được batch size
accumulation_steps = 1

model_name = 'LViT'

train_dataset = './datasets/' + task_name + '/Train_Folder/'
val_dataset   = './datasets/' + task_name + '/Val_Folder/'
test_dataset  = './datasets/' + task_name + '/Test_Folder/'
task_dataset  = './datasets/' + task_name + '/Train_Folder/'
label_plan_csv = './datasets/' + task_name + '/label_plan_100.csv'

session_name       = 'PATCH_' + time.strftime('%m.%d_%Hh%M')
base_save_dir      = '/kaggle/working/'
save_path          = os.path.join(base_save_dir, task_name, model_name, session_name) + os.sep
model_path         = os.path.join(save_path, 'models') + os.sep
tensorboard_folder = os.path.join(save_path, 'tensorboard_logs') + os.sep
logger_path        = os.path.join(save_path, session_name + '.log')
visualize_path     = os.path.join(save_path, 'visualize_val') + os.sep

def get_CTranS_config():
    config = ml_collections.ConfigDict()
    config.transformer = ml_collections.ConfigDict()
    config.KV_size = 960
    config.transformer.num_heads  = 4
    config.transformer.num_layers = 4
    config.expand_ratio = 4
    config.transformer.embeddings_dropout_rate = 0.1
    config.transformer.attention_dropout_rate  = 0.1
    config.transformer.dropout_rate = 0
    config.patch_sizes   = [16, 8, 4, 2]
    config.base_channel  = 64
    config.n_classes     = 1
    return config

# 5. CHECK DATASET & PATCH COUNT

import sys
sys.path.insert(0, '/kaggle/working/LViT')
import os
os.chdir('/kaggle/working/LViT')

from patch_dataset import PatchDataset
from utils import read_text

train_text = read_text('./datasets/BTRXD_fullsize/Train_Folder/Train_text.xlsx')
ds = PatchDataset(
    './datasets/BTRXD_fullsize/Train_Folder',
    train_text, augment=False, seed=666
)
print(f"Train patches: {len(ds)}")

sample = ds[0]
print(f"image: {sample['image'].shape} | label: {sample['label'].shape} | text: {sample['text'].shape}")

# Kiểm tra phân phối positive/negative
import numpy as np
pos_count = sum(1 for p in ds.patches if cv2.imread(p[1], 0)[p[2]:p[2]+224, p[3]:p[3]+224].max() > 0)
print(f"Positive patches: {pos_count} | Negative patches: {len(ds)-pos_count}")
print(f"Ratio pos:neg = 1:{(len(ds)-pos_count)/max(pos_count,1):.1f}")

# 6. TRAIN

In [ ]:
!python train_patch.py

# 7. INFERENCE SLIDING WINDOW

In [ ]:
# Lấy session Phase 2B
import glob

ckpts = glob.glob('/kaggle/working/BTRXD_fullsize/LViT/PATCH_**/models/best_model.pth.tar',
                  recursive=True)
print("Checkpoints found:", ckpts)

SESSION_2B = ckpts[0].split('/')[-4]
print(f"Session 2B: {SESSION_2B}")

In [ ]:
# Ghi test_session vào Config rồi chạy sliding window inference

with open('/kaggle/working/LViT/Config.py', 'a') as f:
    f.write(f'\ntest_session = "{SESSION_2B}"\n')

!python test_sliding_window.py

In [ ]:
# So kết quả Phase 2B vs Phase 2A
result_2b = f'/kaggle/working/BTRXD_fullsize/LViT/{SESSION_2B}/sliding_window_test/results.txt'
print("=== Phase 2B Results ===")
print(open(result_2b).read())

# Xem training log
import csv
log_path = f'/kaggle/working/BTRXD_fullsize/LViT/{SESSION_2B}/training_log.csv'
with open(log_path) as f:
    rows = list(csv.DictReader(f))
print(f"\n{'Epoch':>6} | {'Train Loss':>10} | {'Val Dice':>8} | {'Pred Ratio':>10}")
for r in rows:
    print(f"{r['epoch']:>6} | {r['train_loss']:>10} | {r['val_dice']:>8} | {r['pred_ratio']:>10}")